# Lab L1.2 · Credit-risk feature engineering, leakage audit and a reusable preprocessing pipeline
**Horizon Gulf Bank · Project MIZAN** · CAIP AIP-210 · Certified AI Practitioner · Day 1 · ⏱ 90 min · **Level 1 · From scratch**

Engineer bank-grade credit features, prove which columns leak, split out-of-time, and ship a fitted scikit-learn pipeline that Day 3 and Day 4 reuse.

Follow **L1.2_Lab_Guide.html** step by step. This notebook gives you only the structure — every line of logic is yours. Keep variable names as suggested in the guide so later steps (and later labs) line up.

In [ ]:
# ── Project MIZAN course setup ─ run this cell first ──────────────────────────
from pathlib import Path
import warnings, time
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Locate the course root (the folder that contains data/generate_data.py) from wherever this notebook lives
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "generate_data.py").exists())
DATA = ROOT / "data"          # raw synthetic datasets (read-only)
CKPT = ROOT / "checkpoints"   # hand-off artefacts between labs (lets you start any lab fresh)
CKPT.mkdir(exist_ok=True)
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 160)
plt.rcParams.update({"figure.figsize": (9, 4.5), "axes.grid": True, "grid.alpha": .3})
RNG = 42
print("Course root:", ROOT)

## Step 1 · Load and assign a role to every column

1. Load the file with `application_date` parsed as a date into `loans`.
2. Create lists `ID_COLS`, `TARGET`, `POST_OUTCOME`, `SENSITIVE`.
3. Derive `CANDIDATES` = every other column. Print the overall default rate and the default rate by year.

In [ ]:
# Your code for step 1


## Step 2 · Quantitative leakage screen

1. For each numeric column in `CANDIDATES + POST_OUTCOME`, compute `roc_auc_score(y, x)` on a 50k sample (fill NaN with the median; report `max(auc, 1-auc)`).
2. Put results in a DataFrame sorted descending; flag `suspect = auc > 0.90`.
3. Confirm that only post-outcome columns are flagged.

In [ ]:
# Your code for step 2


## Step 3 · Engineer credit-domain features

1. Write `add_features(d)` that returns a copy with: `loan_to_income` (= amount / annual salary), `installment_to_salary`, `disposable_income_aed`, `log_salary`, `log_loan_amount`, `aecb_missing`, `aecb_band` (Poor <580, Fair <670, Good <740, Very Good <800, Excellent), `thin_file_young` (age < 28 and no bureau score), `employment_years`, `inquiries_x_dq`.
2. Apply it to `loans` → `feat`.
3. Compare default rate by `aecb_band` (should be monotonic).

In [ ]:
# Your code for step 3


## Step 4 · Out-of-time train / validation / test split

1. Add a `split` column: `train` < 2023-07-01 ≤ `valid` < 2024-07-01 ≤ `test`.
2. Print size and default rate per split.
3. Explain in a markdown cell why a random split would be optimistic here.

In [ ]:
# Your code for step 4


## Step 5 · Build the preprocessing ColumnTransformer

1. Define `NUM_LOG` (money columns: log1p then scale), `NUM` (other numerics: median-impute + indicator, scale), `CAT` (one-hot, unknown-safe, rare → infrequent), `ORD` (`employer_category`, ordered), `BIN` (0/1 flags passthrough).
2. Build `preprocessor` and fit it on the **train** split only.
3. Transform the train split and print the output shape and the first 15 feature names.

In [ ]:
# Your code for step 5


## Step 6 · Prove the pipeline is honest: baseline model with and without leakage

1. Wrap `preprocessor` + `LogisticRegression(max_iter=2000)` in a Pipeline; fit on train, report validation AUC.
2. Repeat with `max_days_past_due` and `collection_calls` added as extra numeric inputs. Report both AUCs side by side.

In [ ]:
# Your code for step 6


## Step 7 · Feature-selection fundamentals

1. Using the transformed training matrix, list columns with variance < 0.01.
2. Find feature pairs with |correlation| > 0.85.
3. Compute `mutual_info_classif` on a 40k-row sample and show the top 12 features.

In [ ]:
# Your code for step 7


## Step 8 · Persist the data contract for Day 3 & Day 4

1. Save `feat` (all rows, with `split`) to `CKPT/loan_features.parquet`.
2. Save the fitted `preprocessor` with `joblib.dump` to `CKPT/loan_preprocessor.joblib`.
3. Write `loan_feature_spec.json` with keys `target`, `num_log`, `num`, `cat`, `ord`, `bin`, `sensitive`, `post_outcome`, `honest_valid_auc`.
4. Reload the joblib file and assert it transforms the validation split to the same width.

In [ ]:
# Your code for step 8
